## installs, setup

In [ ]:
!pip -q install openai pydantic tqdm pillow

import os
import re
import json
import time
import base64
import textwrap
from PIL import Image, ImageDraw, ImageFont
from openai import OpenAI
from pathlib import Path
from typing import List, Optional, Literal

from tqdm import tqdm
from pydantic import BaseModel, Field
from openai import OpenAI




## mount to drive, import API key

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OpenAI client initialized.


## define structure

In [ ]:

CardType = Literal["robot","automaton","machine","android","golem","mechanism","other"]

class SourceSpan(BaseModel):
    source_path: str
    chunk_index: int
    evidence_quote: str
    location_hint: str

class Ability(BaseModel):
    name: str
    effect: str

class AutomataCard(BaseModel):
    card_id: str
    title: str
    card_type: CardType
    is_explicit_in_text: bool
    entity_name_in_text: str
    summary: str
    era_or_context: str
    themes: List[str]
    abilities: List[Ability]
    limitations: List[str]
    flavor_quote: str
    art_prompt: str
    safety_notes: str
    sources: List[SourceSpan]

class ExtractionResult(BaseModel):
    source_path: str
    chunk_index: int
    found_any: bool
    cards: List[AutomataCard]


In [ ]:
MODEL = "gpt-4o"
SYSTEM_PROMPT = """You extract robot/machine/automaton-like entities from a text.

Requirements:
- Output MUST match the schema exactly.
- Only extract entities supported by the provided text.
- If none exist: found_any=false and cards=[]
- evidence_quote must be copied exactly from the text.
- If a detail is unknown, use 'unknown' or '' rather than inventing it.
"""

# 1) Pick one text
source_path = Path("/content/drive/MyDrive/complit126x/inputs/texts/kj-bible/Ezekiel.txt")
text = source_path.read_text(encoding="utf-8", errors="replace")

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"SOURCE_PATH: {source_path.as_posix()}\n\nTEXT:\n{text}"},
    ],
    text_format=ExtractionResult,
)

result = response.output_parsed

# Print the structured JSON
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))
text = source_path.read_text(encoding="utf-8", errors="replace")


{
  "source_path": "/content/drive/MyDrive/complit126x/inputs/texts/kj-bible/Ezekiel.txt",
  "chunk_index": 0,
  "found_any": true,
  "cards": [
    {
      "card_id": "living_creatures",
      "title": "The Four Living Creatures and Wheels",
      "card_type": "other",
      "is_explicit_in_text": true,
      "entity_name_in_text": "living creatures and wheels",
      "summary": "The vision includes four living creatures with a man's likeness, each having four faces and four wings. Accompanying the creatures are wheels, described as having the appearance of a wheel within a wheel, that move with the creatures and are full of eyes.",
      "era_or_context": "Prophet Ezekiel's Vision",
      "themes": [
        "Vision",
        "Symbolism",
        "Divine Presence"
      ],
      "abilities": [
        {
          "name": "Movement Coordination",
          "effect": "The wheels moved in conjunction with the living creatures, directed by a spirit."
        },
        {
          "name"

## make a card

In [ ]:

OUT_DIR = Path("/content/drive/MyDrive/complit126x/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CARD_W, CARD_H = 750, 1050
ART_H = 520
PADDING = 30

def load_font(size):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()

TITLE_FONT = load_font(44)
BODY_FONT  = load_font(24)
SMALL_FONT = load_font(20)

def safe_filename(s: str) -> str:
    s = s.strip().lower()
    s = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in s)
    return s[:80] or "card"

def wrap(draw, text, font, max_width):
    words = text.split()
    lines, line = [], ""
    for w in words:
        test = (line + " " + w).strip()
        if draw.textlength(test, font=font) <= max_width:
            line = test
        else:
            if line:
                lines.append(line)
            line = w
    if line:
        lines.append(line)
    return lines


def generate_art_png(prompt: str, out_path, size: str = "1024x1024"):
    """
    Generate art using DALL·E 3
    """
    img_resp = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,
        quality="standard",
        response_format="b64_json",
    )

    b64 = img_resp.data[0].b64_json
    out_path.write_bytes(base64.b64decode(b64))

def build_card_png(card: dict, art_path: Path, out_path: Path):
    """
    Compose a card: art at top, text below.
    """
    canvas = Image.new("RGB", (CARD_W, CARD_H), "white")
    draw = ImageDraw.Draw(canvas)

    art = Image.open(art_path).convert("RGB")
    art = art.resize((CARD_W, ART_H))
    canvas.paste(art, (0, 0))

    draw.rectangle([0, ART_H, CARD_W, ART_H + 80], fill="white")
    title = card.get("title", "Untitled")
    draw.text((PADDING, ART_H + 15), title, font=TITLE_FONT, fill="black")

    y = ART_H + 95

    def section(label, content):
        nonlocal y
        if not content:
            return
        draw.text((PADDING, y), label, font=SMALL_FONT, fill="black")
        y += 26
        max_w = CARD_W - 2 * PADDING
        lines = wrap(draw, str(content), BODY_FONT, max_w)
        for ln in lines[:8]:  # keep it sane
            draw.text((PADDING, y), ln, font=BODY_FONT, fill="black")
            y += 30
        y += 12

    section("Type", card.get("card_type", "other"))
    section("Summary", card.get("summary", ""))
    themes = ", ".join(card.get("themes", []))
    section("Themes", themes)

    abilities = card.get("abilities", [])
    if abilities:
        draw.text((PADDING, y), "Abilities", font=SMALL_FONT, fill="black")
        y += 26
        for a in abilities[:3]:
            line = f"- {a.get('name','')}: {a.get('effect','')}"
            lines = wrap(draw, line, BODY_FONT, CARD_W - 2 * PADDING)
            for ln in lines[:3]:
                draw.text((PADDING, y), ln, font=BODY_FONT, fill="black")
                y += 30
            y += 6
        y += 8

    fq = card.get("flavor_quote", "")
    if fq:
        section("Flavor", f"“{fq}”")

    sources = card.get("sources", [])
    if sources:
        src0 = sources[0]
        footer = f"{src0.get('source_path','')} | {src0.get('location_hint','')}"
        draw.text((PADDING, CARD_H - 35), footer[:120], font=SMALL_FONT, fill="black")

    canvas.save(out_path)

# --- Run: generate art + build cards ---
cards = result.model_dump().get("cards", [])
if not cards:
    print("No cards found. Nothing to render.")
else:
    for c in cards:
        card_id = c.get("card_id") or safe_filename(c.get("title", "card"))
        slug = safe_filename(card_id)

        card_json_path  = OUT_DIR / f"card_{slug}.json"
        art_png_path    = OUT_DIR / f"card_{slug}_art.png"
        final_png_path  = OUT_DIR / f"card_{slug}_final.png"

        # Save the card JSON
        card_json_path.write_text(json.dumps(c, ensure_ascii=False, indent=2), encoding="utf-8")

        # Generate art (prompt is already in the structured output)
        art_prompt = c.get("art_prompt", "").strip()
        if not art_prompt:
            # fallback prompt if model returned empty (shouldn't, but humans love edge cases)
            art_prompt = f"Illustration of {c.get('title','an automaton')} in an ancient manuscript style."

        generate_art_png(art_prompt, art_png_path)

        # Compose the final card image
        build_card_png(c, art_png_path, final_png_path)

        print("Wrote:", final_png_path)


Wrote: /content/drive/MyDrive/complit126x/outputs/card_living_creatures_final.png
